In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/Users/anuragabcr@gmail.com/learning_spark/fmcg_atlikon_project/03_utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema, s3_bucket)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders/landing", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://{s3_bucket}/{data_source}/*.csv'
print(base_path)

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.orders"
silver_table = f"{catalog}.{silver_schema}.orders"
gold_table = f"{catalog}.{gold_schema}.sb_fact_orders"

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
display(df.limit(10))

In [0]:
display(df.count())

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{bronze_schema}.orders")

In [0]:
files = dbutils.fs.ls(f's3://{s3_bucket}/{data_source}')
for file in files:
    dbutils.fs.mv(
        file.path,
        f's3://{s3_bucket}/orders/processed/{file.name}',
        True
    )

In [0]:
silver_df = spark.sql("SELECT * FROM fmcg.bronze.orders")
display(silver_df.limit(10))

In [0]:
silver_df = silver_df.filter(F.col("order_qty").isNotNull())

silver_df = silver_df.withColumn("order_id",
                                 F.when(F.col("order_id").rlike("^[0-9]+$"), F.col("customer_id"))
                                 .otherwise("999999")
                                 .cast("string")
                                 )

silver_df = silver_df.withColumn("order_placement_date",
                                 F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
                                 )

silver_df = silver_df.withColumn("order_placement_date",
                                 F.coalesce(
                                    F.try_to_date("order_placement_date", "yyyy/MM/dd"),
                                    F.try_to_date("order_placement_date", "dd-MM-yyyy"),
                                    F.try_to_date("order_placement_date", "dd/MM/yyyy"),
                                    F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
                                )
                                )

silver_df = silver_df.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

silver_df = silver_df.withColumn("product_id", F.col("product_id").cast("string"))

In [0]:
display(silver_df.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
    )
)

In [0]:
display(silver_df.filter(F.col("order_placement_date").isNull()).count())

In [0]:
products_df = spark.table("fmcg.silver.products")
display(products_df.limit(10))

In [0]:
silver_joined_df = silver_df.join(products_df, on="product_id", how="inner").select(silver_df["*"], products_df["product_code"])
display(silver_joined_df.limit(10))

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    silver_joined_df.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(silver_joined_df.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

In [0]:
df_child.count()

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()